In [1]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import numpy as np
import optuna

import sys
from pathlib import Path
import os

from torch.utils.tensorboard import SummaryWriter

import albumentations as A
from albumentations.pytorch import ToTensorV2


/home/jorge/miniconda3/envs/image-recon-dl-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jorge/miniconda3/envs/image-recon-dl-env/lib/python3.10/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.0 (you have 1.4.21). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
# Semilla para reproducibilidad de los experimentos
random.seed(42)
np.random.seed(42)
torch.manual_seed(42);

In [3]:
# Me aseguro de que el directorio raíz del proyecto esté en el sys.path
project_root = Path(os.path.abspath("")).parent

# Añado el directorio raíz al sys.path si no está ya presente
if project_root not in sys.path:
    sys.path.append(str(project_root))

In [4]:
# Importo las funciones de configuración
from src.config import processed_data_dir, models_dir, reports_dir, load_config
from src.utils.datasets import CustomDataset_2_2

from src.models.convolutional_autoencoder_model.model import ConvolutionalAutoencoder
from src.models.convolutional_autoencoder_model.optuna_aux import objective

from src.models.convolutional_autoencoder_model.train_batch import train_batch
from src.models.convolutional_autoencoder_model.validate_batch import compute_val_metrics

Current working directory: /home/jorge/development/WasteImageReconstructionDL/notebooks
Loading configuration from /home/jorge/development/WasteImageReconstructionDL/src/config.yaml


In [5]:
# Cargo la configuración 
config = load_config()


Loading configuration from /home/jorge/development/WasteImageReconstructionDL/src/config.yaml


In [6]:
# Transformación de las imágenes
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

In [7]:
# Definir el pipeline de augmentación
augmentation_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.OneOf([
        A.Rotate(limit=(0, 0), p=0.33),
        A.Rotate(limit=(180, 180), p=0.33),
        A.Compose([A.Rotate(limit=(180, 180), p=1.0), A.HorizontalFlip(p=1.0)], p=0.34)
    ], p=1.0)
])

In [8]:
# Ruta de los datos
#original_dir = processed_data_dir() / 'dataset_final'
final_data_dir = processed_data_dir()

In [9]:
# Definir la ruta de los archivos CSV
train_csv = os.path.join(final_data_dir, 'train.csv')
val_csv = os.path.join(final_data_dir, 'val.csv')

In [10]:
# Crear el dataset
train_dataset = CustomDataset_2_2(
    csv_file=os.path.join(final_data_dir, 'train.csv'),
    transform=transform,
    augmentation_pipeline=augmentation_pipeline,
    use_augmentation=True  # Activar Data Augmentation para el entrenamiento y probar con imágenes aumentadas
)

val_dataset = CustomDataset_2_2(
    csv_file=os.path.join(final_data_dir, 'val.csv'),
    transform=transform,
    use_augmentation=False  # No activar Data Augmentation para el conjunto de validación
)

In [11]:
# Crear el dataset
#dataset = CustomDataset(original_dir=original_dir, transform=transform)

In [12]:
# Si tenemos disponible GPU, lo usamos
# Chequeamos si tenemos disponible GPU (CUDA)
if torch.cuda.is_available():
    device = "cuda"
# Chequeamos si tenemos disponible aceleración por hardware en un chip de Apple (MPS)
elif torch.backends.mps.is_available():
    device = "mps"
# Por defecto usamos CPU
else:
    device = "cpu"

In [13]:
# Dividir el dataset en train, validation y test
#train_size = int(0.8 * len(dataset))
#val_size = int(0.1 * len(dataset))
#test_size = len(dataset) - train_size - val_size
#train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

In [14]:
# Función para entrenar un batch
# def train_batch2(model, train_loader, criterion, optimizer, device):
#     model.train()
#     train_loss = 0.0
#     for original_image, _ in train_loader:
#         original_image = original_image.to(device)
        
#         optimizer.zero_grad()
#         outputs = model(original_image)
        
#         loss = criterion(outputs, original_image)
        
#         loss.backward()
#         optimizer.step()
#         train_loss += loss.item()
        
#     train_loss /= len(train_loader)
#     return train_loss

In [15]:
# Función para calcular métricas de validación
# @torch.no_grad()
# def compute_val_metrics2(model, val_loader, criterion, device):
#     model.eval()
#     val_loss = 0.0
#     val_psnr = 0.0
#     val_ssim = 0.0
#     compression_ratios = []
#     for original_image, _ in val_loader:
#         original_image = original_image.to(device)
        
#         encoded = model.compress(original_image)
#         outputs = model.decompress(encoded)
        
#         loss = criterion(outputs, original_image)
        
#         val_loss += loss.item()
        
#         # Convertir las imágenes a numpy para calcular las métricas
#         original_np = original_image.cpu().numpy().transpose(0, 2, 3, 1)
#         reconstructed_np = outputs.cpu().detach().numpy().transpose(0, 2, 3, 1)
        
#         for orig, rec in zip(original_np, reconstructed_np):
#             psnr_value, ssim_value = calculate_metrics(orig, rec)
#             val_psnr += psnr_value
#             val_ssim += ssim_value

#         # Calcular la razón de compresión
#         batch_size, latent_channels, latent_height, latent_width = encoded.size()
#         latent_size = latent_height * latent_width * latent_channels
#         original_size = 256 * 256 * 3
#         compression_ratio = calculate_compression_ratio(original_size, latent_size)
#         compression_ratios.append(compression_ratio)

#     val_loss /= len(val_loader)
#     val_psnr /= len(val_loader.dataset)
#     val_ssim /= len(val_loader.dataset)
#     avg_compression_ratio = sum(compression_ratios) / len(compression_ratios)
    
#     return val_loss, val_psnr, val_ssim, avg_compression_ratio

In [16]:
# Definir la función objetivo para Optuna
# def objective2(trial):
#     # Definir los hiperparámetros a optimizar
#     #encoder_filters = [trial.suggest_int(f"encoder_filters_{i}", 64, 1024, step=64) for i in range(6)]
#     #decoder_filters = list(reversed(encoder_filters))

#     encoder_filters = [64, 128, 256, 512, 1024, 2048]
#     decoder_filters = list(reversed(encoder_filters))
#     lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
#     weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
#     batch_size = trial.suggest_int("batch_size", 8, 128, step=8) 
        
#     # Imprimir los hiperparámetros usados en el trial actual
#     print(f"Trial {trial.number}:")
#     print(f"  encoder_filters: {encoder_filters}")
#     print(f"  decoder_filters: {decoder_filters}")
#     print(f"  lr: {lr}")
#     print(f"  weight_decay: {weight_decay}")
#     print(f"  batch_size: {batch_size}")

#     # Definir el modelo
#     model = ConvolutionalAutoencoder(encoder_filters, decoder_filters).to(device)
    
#     # Definir el criterio y el optimizador
#     criterion = nn.MSELoss()
#     optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
#     scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

#     # Crear los dataloaders con el batch_size actual
#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
#     val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

#     # Inicializar TensorBoard
#     nombre_modelo = 'convolutional_autoencoder_model_optuna_2_PRES'
#     log_dir = reports_dir() / "logs"/ nombre_modelo / f"trial_{trial.number}"
#     print(f"Guardando logs en {log_dir}")
#     os.makedirs(log_dir, exist_ok=True)
#     writer = SummaryWriter(log_dir=str(log_dir))
    
#     # Entrenar el modelo
#     num_epochs = 30
#     best_val_loss = float('inf')
#     early_stop_counter = 0
#     early_stop_patience = 8
    
#     for epoch in range(num_epochs):
#         train_loss = train_batch(model, train_loader, criterion, optimizer, device)
        
#         val_loss, val_psnr, val_ssim, avg_compression_ratio = compute_val_metrics(model, val_loader, criterion, device)
        
#         scheduler.step(val_loss)
        
#         # Guardar los valores en TensorBoard y imprimir
#         writer.add_scalar(f'trial_{trial.number}/Loss/train', train_loss, epoch)
#         writer.add_scalar(f'trial_{trial.number}/Loss/val', val_loss, epoch)
#         writer.add_scalar(f'trial_{trial.number}/val/PSNR', val_psnr, epoch)
#         writer.add_scalar(f'trial_{trial.number}/val/SSIM', val_ssim, epoch)
#         writer.add_scalar(f'trial_{trial.number}/val/Compression_Ratio', avg_compression_ratio, epoch)

#         # Registrar histogramas de los parámetros del modelo
#         for name, param in model.named_parameters():
#             writer.add_histogram(f'{name}', param, epoch)

#         print(f'Trial {trial.number}, Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, '
#               f'Val PSNR: {val_psnr:.4f}, Val SSIM: {val_ssim:.4f}, Compression Ratio: {avg_compression_ratio:.2f}')

#         if val_loss < best_val_loss:
#             best_val_loss = val_loss
#             early_stop_counter = 0
#         else:
#             early_stop_counter += 1
        
#         if early_stop_counter >= early_stop_patience:
#             break
#     # Justo antes de writer.close()
#     hparams = {
#         "encoder_filters": str(encoder_filters),  # Convertir a str
#         "decoder_filters": str(decoder_filters),  # Convertir a str
#         "lr": lr,
#         "weight_decay": weight_decay,
#         "batch_size": batch_size,
#         "trial_number": trial.number,
#     }

#     metrics = {
#         "best_val_loss": best_val_loss,
#         "val_psnr": val_psnr,
#         "val_ssim": val_ssim,
#         "avg_compression_ratio": avg_compression_ratio,
#     }

#     writer.add_hparams(hparams, metrics)
#     writer.close()  # Cerrar TensorBoard
#     return best_val_loss


In [17]:
# Defino la función objetivo para Optuna
def objective3(trial, train_dataset, val_dataset, device):
    # Defino los hiperparámetros a optimizar
    encoder_filters = [64, 128, 256, 512, 1024, 2048]
    decoder_filters = list(reversed(encoder_filters))
    #lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    lr = trial.suggest_float("lr", 5e-5, 1e-4, log=False)

    #weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 4, 12, step=8) 
        
    # Imprimo los hiperparámetros usados en el trial actual
    print(f"Trial {trial.number}:")
    print(f"  encoder_filters: {encoder_filters}")
    print(f"  decoder_filters: {decoder_filters}")
    print(f"  lr: {lr}")
    #print(f"  weight_decay: {weight_decay}")
    print(f"  batch_size: {batch_size}")

    # Defino el modelo
    model = ConvolutionalAutoencoder(encoder_filters, decoder_filters).to(device)
    
    # Defino el criterio y el optimizador
    criterion = nn.MSELoss()
    #optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

    # Crear los dataloaders con el batch_size actual
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # Inicializar TensorBoard
    nombre_modelo = 'convolutional_autoencoder_model_optuna_2_PRES'
    log_dir = reports_dir() / "logs"/ nombre_modelo / f"trial_{trial.number}"
    print(f"Guardando logs en {log_dir}")
    os.makedirs(log_dir, exist_ok=True)
    writer = SummaryWriter(log_dir=str(log_dir))
    
    # Entrenar el modelo
    num_epochs = 30
    best_val_loss = float('inf')
    early_stop_counter = 0
    early_stop_patience = 8
    train_losses = []
    val_losses = []
    val_psnr_values = []
    val_ssim_values = []
    compression_ratios = []
    
    for epoch in range(num_epochs):
        train_loss = train_batch(model, train_loader, criterion, optimizer, device)
        train_losses.append(train_loss)
        
        val_loss, val_psnr, val_ssim, avg_compression_ratio = compute_val_metrics(model, val_loader, criterion, device)
        val_losses.append(val_loss)
        val_psnr_values.append(val_psnr)
        val_ssim_values.append(val_ssim)
        compression_ratios.append(avg_compression_ratio)
        
        # Guardar los valores en TensorBoard y imprimir
        writer.add_scalar(f'trial_{trial.number}/Loss/train', train_loss, epoch)
        writer.add_scalar(f'trial_{trial.number}/Loss/val', val_loss, epoch)
        writer.add_scalar(f'trial_{trial.number}/val/PSNR', val_psnr, epoch)
        writer.add_scalar(f'trial_{trial.number}/val/SSIM', val_ssim, epoch)
        writer.add_scalar(f'trial_{trial.number}/val/Compression_Ratio', avg_compression_ratio, epoch)

        scheduler.step(val_loss)

        # Registrar histogramas de los parámetros del modelo
        for name, param in model.named_parameters():
            writer.add_histogram(f'{name}', param, epoch)

        print(f'Trial {trial.number}, Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, '
              f'Val PSNR: {val_psnr:.4f}, Val SSIM: {val_ssim:.4f}, Compression Ratio: {avg_compression_ratio:.2f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stop_counter = 0
        else:
            early_stop_counter += 1
        
        if early_stop_counter >= early_stop_patience:
            break
    # Justo antes de writer.close()
    hparams = {
        "encoder_filters": str(encoder_filters),  # Convertir a str
        "decoder_filters": str(decoder_filters),  # Convertir a str
        "lr": lr,
        #"weight_decay": weight_decay,
        "batch_size": batch_size,
        "trial_number": trial.number,
    }

    metrics = {
        "best_val_loss": best_val_loss,
        "val_psnr": val_psnr,
        "val_ssim": val_ssim,
        "avg_compression_ratio": avg_compression_ratio,
    }

    writer.add_hparams(hparams, metrics)
    writer.close()  # Cerrar TensorBoard
    return best_val_loss

In [18]:
%load_ext tensorboard
%tensorboard --logdir ../reports/logs/convolutional_autoencoder_model_optuna_2_PRES


In [19]:
# Inicializar el estudio de Optuna
study = optuna.create_study(direction="minimize")
#study.optimize(objective, n_trials=30)
#study.optimize(objective(train_dataset=train_dataset, val_dataset=val_dataset, device=device), n_trials=30)
# Llamar a study.optimize pasando la función objective como argumento
study.optimize(lambda trial: objective3(trial, train_dataset=train_dataset, val_dataset=val_dataset, device=device), n_trials=30)

# Mostrar los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", study.best_trial.params)

[I 2025-01-13 19:09:55,400] A new study created in memory with name: no-name-922795a5-5013-40d4-a1ce-01721ab1db63


Trial 0:
  encoder_filters: [64, 128, 256, 512, 1024, 2048]
  decoder_filters: [2048, 1024, 512, 256, 128, 64]
  lr: 5.41080394288781e-05
  batch_size: 4
Guardando logs en /home/jorge/development/WasteImageReconstructionDL/reports/logs/convolutional_autoencoder_model_optuna_2_PRES/trial_0
Trial 0, Epoch [1/30], Train Loss: 0.0424, Val Loss: 0.0281, Val PSNR: 13.8934, Val SSIM: 0.1903, Compression Ratio: 6.00
Trial 0, Epoch [2/30], Train Loss: 0.0199, Val Loss: 0.0170, Val PSNR: 15.6025, Val SSIM: 0.2354, Compression Ratio: 6.00
Trial 0, Epoch [3/30], Train Loss: 0.0155, Val Loss: 0.0145, Val PSNR: 16.8113, Val SSIM: 0.2401, Compression Ratio: 6.00
Trial 0, Epoch [4/30], Train Loss: 0.0137, Val Loss: 0.0132, Val PSNR: 17.6968, Val SSIM: 0.2475, Compression Ratio: 6.00
Trial 0, Epoch [5/30], Train Loss: 0.0131, Val Loss: 0.0127, Val PSNR: 17.6516, Val SSIM: 0.2541, Compression Ratio: 6.00
Trial 0, Epoch [6/30], Train Loss: 0.0123, Val Loss: 0.0118, Val PSNR: 18.4136, Val SSIM: 0.2614, Co

[I 2025-01-13 19:17:30,097] Trial 0 finished with value: 0.007969548541586846 and parameters: {'lr': 5.41080394288781e-05, 'batch_size': 4}. Best is trial 0 with value: 0.007969548541586846.


Trial 0, Epoch [30/30], Train Loss: 0.0078, Val Loss: 0.0080, Val PSNR: 21.0710, Val SSIM: 0.3355, Compression Ratio: 6.00
Trial 1:
  encoder_filters: [64, 128, 256, 512, 1024, 2048]
  decoder_filters: [2048, 1024, 512, 256, 128, 64]
  lr: 8.30786803443442e-05
  batch_size: 12
Guardando logs en /home/jorge/development/WasteImageReconstructionDL/reports/logs/convolutional_autoencoder_model_optuna_2_PRES/trial_1
Trial 1, Epoch [1/30], Train Loss: 0.0406, Val Loss: 0.0348, Val PSNR: 12.6177, Val SSIM: 0.2120, Compression Ratio: 6.00
Trial 1, Epoch [2/30], Train Loss: 0.0227, Val Loss: 0.0173, Val PSNR: 15.6904, Val SSIM: 0.2321, Compression Ratio: 6.00
Trial 1, Epoch [3/30], Train Loss: 0.0160, Val Loss: 0.0147, Val PSNR: 16.6433, Val SSIM: 0.2379, Compression Ratio: 6.00
Trial 1, Epoch [4/30], Train Loss: 0.0144, Val Loss: 0.0138, Val PSNR: 17.3540, Val SSIM: 0.2466, Compression Ratio: 6.00
Trial 1, Epoch [5/30], Train Loss: 0.0137, Val Loss: 0.0132, Val PSNR: 17.6507, Val SSIM: 0.2506, 

[I 2025-01-13 19:22:12,985] Trial 1 finished with value: 0.008871027710847556 and parameters: {'lr': 8.30786803443442e-05, 'batch_size': 12}. Best is trial 0 with value: 0.007969548541586846.


...

Trial 27, Epoch [30/30], Train Loss: 0.0077, Val Loss: 0.0079, Val PSNR: 21.0334, Val SSIM: 0.3369, Compression Ratio: 6.00
Trial 28:
  encoder_filters: [64, 128, 256, 512, 1024, 2048]
  decoder_filters: [2048, 1024, 512, 256, 128, 64]
  lr: 7.428833783357554e-05
  batch_size: 4
Guardando logs en /home/jorge/development/WasteImageReconstructionDL/reports/logs/convolutional_autoencoder_model_optuna_2_PRES/trial_28
Trial 28, Epoch [1/30], Train Loss: 0.0353, Val Loss: 0.0188, Val PSNR: 14.8251, Val SSIM: 0.2287, Compression Ratio: 6.00
Trial 28, Epoch [2/30], Train Loss: 0.0168, Val Loss: 0.0156, Val PSNR: 16.3308, Val SSIM: 0.2403, Compression Ratio: 6.00
Trial 28, Epoch [3/30], Train Loss: 0.0143, Val Loss: 0.0139, Val PSNR: 16.9878, Val SSIM: 0.2479, Compression Ratio: 6.00
Trial 28, Epoch [4/30], Train Loss: 0.0131, Val Loss: 0.0124, Val PSNR: 17.9952, Val SSIM: 0.2537, Compression Ratio: 6.00
Trial 28, Epoch [5/30], Train Loss: 0.0122, Val Loss: 0.0120, Val PSNR: 18.2616, Val SSIM: 

[I 2025-01-13 22:31:19,555] Trial 28 finished with value: 0.007910838678556804 and parameters: {'lr': 7.428833783357554e-05, 'batch_size': 4}. Best is trial 26 with value: 0.007682575739454478.


Trial 28, Epoch [30/30], Train Loss: 0.0078, Val Loss: 0.0079, Val PSNR: 20.9879, Val SSIM: 0.3363, Compression Ratio: 6.00
Trial 29:
  encoder_filters: [64, 128, 256, 512, 1024, 2048]
  decoder_filters: [2048, 1024, 512, 256, 128, 64]
  lr: 8.376672272631378e-05
  batch_size: 4
Guardando logs en /home/jorge/development/WasteImageReconstructionDL/reports/logs/convolutional_autoencoder_model_optuna_2_PRES/trial_29
Trial 29, Epoch [1/30], Train Loss: 0.0310, Val Loss: 0.0177, Val PSNR: 15.6331, Val SSIM: 0.2331, Compression Ratio: 6.00
Trial 29, Epoch [2/30], Train Loss: 0.0165, Val Loss: 0.0158, Val PSNR: 16.0385, Val SSIM: 0.2440, Compression Ratio: 6.00
Trial 29, Epoch [3/30], Train Loss: 0.0143, Val Loss: 0.0134, Val PSNR: 17.4950, Val SSIM: 0.2505, Compression Ratio: 6.00
Trial 29, Epoch [4/30], Train Loss: 0.0133, Val Loss: 0.0128, Val PSNR: 17.6073, Val SSIM: 0.2539, Compression Ratio: 6.00
Trial 29, Epoch [5/30], Train Loss: 0.0121, Val Loss: 0.0120, Val PSNR: 18.1631, Val SSIM: 

[I 2025-01-13 22:38:46,891] Trial 29 finished with value: 0.008005047391634434 and parameters: {'lr': 8.376672272631378e-05, 'batch_size': 4}. Best is trial 26 with value: 0.007682575739454478.


Trial 29, Epoch [30/30], Train Loss: 0.0078, Val Loss: 0.0081, Val PSNR: 20.7674, Val SSIM: 0.3362, Compression Ratio: 6.00
Mejores hiperparámetros: {'lr': 8.149533585849922e-05, 'batch_size': 4}


In [20]:
print("Mejores hiperparámetros:", study.best_trial.params)

Mejores hiperparámetros: {'lr': 8.149533585849922e-05, 'batch_size': 4}


In [21]:
print("Mejores hiperparámetros:", study.best_trial)

Mejores hiperparámetros: FrozenTrial(number=26, state=TrialState.COMPLETE, values=[0.007682575739454478], datetime_start=datetime.datetime(2025, 1, 13, 22, 8, 56, 961380), datetime_complete=datetime.datetime(2025, 1, 13, 22, 16, 24, 807692), params={'lr': 8.149533585849922e-05, 'batch_size': 4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'lr': FloatDistribution(high=0.0001, log=False, low=5e-05, step=None), 'batch_size': IntDistribution(high=12, log=False, low=4, step=8)}, trial_id=26, value=None)


In [ ]:
#Mejores hiperparámetros: {'lr': 8.149533585849922e-05, 'batch_size': 4}